In [ ]:
import os
import math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tslearn.clustering import TimeSeriesKMeans

In [ ]:
path = './data/nifty500_5yr_daily/'
ex_rate_path = './data/usd_to_inr/'
price_col = 'avg' # CH_TRADE_AVG_PRICE
date_col = 'date' # CH_TIMESTAMP

In [ ]:
ex_rate_files = sorted(set(os.listdir(ex_rate_path)))
ex_rate_df_list = []
for file in ex_rate_files:
    df = pd.read_csv(ex_rate_path + file)
    ex_rate_df_list.append(df)

df_ex_rate = pd.concat(ex_rate_df_list)
df_ex_rate['USD_to_INR'] = df_ex_rate.apply(lambda row : int((row['Open'] + row['High'] + row['Low'] + row['Close'])/4), axis=1)
df_ex_rate['Date'] = pd.to_datetime(df_ex_rate['Date']).astype(str)
df_ex_rate.drop(columns=['Open', 'High', 'Low', 'Close'], inplace=True)
df_ex_rate

In [ ]:
files = sorted(set(os.listdir(path)))
files = [file for file in files if file.endswith('csv')]
symbols = [file.split('.')[0] for file in files]

mySeries = []
namesofMySeries = []
for file, symbol in zip(files, symbols) :
    df = pd.read_csv(path+file)[[date_col, price_col]]
    df.drop_duplicates(date_col, inplace=True)
    df = df.merge(df_ex_rate, left_on=date_col, right_on='Date', how='left')
    df['price_in_usd'] = df[price_col] / df['USD_to_INR']
    df.drop(columns=['Date', price_col, 'USD_to_INR'], inplace=True)
    df.set_index(date_col, inplace=True)
    df.sort_index(inplace=True)
    df = df.rolling(window=30, center=True).mean().dropna().astype(int)
    df = df.tail(1000)
    if len(df) < 846 :
        print(f'dropping {symbol} as it has only {len(df)} days')
    else:
        factor_rs100 = df.iloc[0,0] / 100 # converting to "Rs. 100 invested in ..." format
        df['price_in_usd'] = df['price_in_usd'] / factor_rs100
        mySeries.append(df)
        namesofMySeries.append(symbol)

In [ ]:
n_rows = int(np.ceil(len(mySeries)/4))

fig, axs = plt.subplots(n_rows,4,figsize=(n_rows/8,n_rows))
fig.suptitle('Series')
for i in range(n_rows):
    for j in range(4):
        if i*4+j+1>len(mySeries): # pass the others that we can't fill
            continue
        axs[i, j].plot(mySeries[i*4+j].values)
        axs[i, j].set_title(namesofMySeries[i*4+j])
fig.tight_layout()
plt.savefig('nifty_500_5yr_30D_mov_avg_infl_adj_usd.png')
plt.show()

In [ ]:
som_x = som_y = math.ceil(math.sqrt(math.sqrt(len(mySeries))))

In [ ]:
cluster_count = 100

km = TimeSeriesKMeans(n_clusters=cluster_count, metric="dtw")

labels = km.fit_predict(mySeries)

In [ ]:
pd.DataFrame(labels,columns=['cluster_labels']).to_csv('./data/cluster_labels.csv', index=None)

In [ ]:
plot_count = math.ceil(math.sqrt(cluster_count))

fig, axs = plt.subplots(plot_count,plot_count,figsize=(50,50))
fig.suptitle('Clusters')
row_i=0
column_j=0
# For each label there is,
# plots every series with that label
for label in set(labels):
    cluster = []
    for i in range(len(labels)):
            if(labels[i]==label):
                axs[row_i, column_j].plot(mySeries[i],c="gray",alpha=0.4)
                cluster.append(mySeries[i])
    if len(cluster) > 0:
        axs[row_i, column_j].plot(np.average(np.vstack(cluster),axis=0),c="red")
    axs[row_i, column_j].set_title("Cluster "+str(row_i*som_y+column_j))
    column_j+=1
    if column_j%plot_count == 0:
        row_i+=1
        column_j=0
fig.tight_layout()
plt.show()

In [ ]:
# inspect members of any specific cluster 
for i, j in zip(namesofMySeries, labels):
    if j == 99: # change cluster label here as required
        print(i, j)